# YOLOv8 Toddler Detection — Fine-Tuning
**Before running:** Upload `data.zip` to your Google Drive root (My Drive).

In [ ]:
# 1. Check GPU
!nvidia-smi

In [ ]:
# 2. Install ultralytics
!pip install ultralytics -q

In [ ]:
# 3. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 4. Unzip dataset
import zipfile, os

ZIP_PATH    = '/content/drive/MyDrive/data.zip'   # change if you put it elsewhere
EXTRACT_DIR = '/content/dataset'

with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall(EXTRACT_DIR)

# Show structure
for root, dirs, files in os.walk(EXTRACT_DIR):
    depth = root.replace(EXTRACT_DIR, '').count(os.sep)
    if depth < 3:
        print(' ' * depth * 2 + os.path.basename(root) + '/')
        if depth == 2:
            print(' ' * (depth+1) * 2 + f'({len(files)} files)')

In [ ]:
# 5. Merge ALL images then re-split 70% train / 15% val / 15% test
import shutil, random
from pathlib import Path

BASE = Path(EXTRACT_DIR) / 'detect_child'

SPLITS = ['train', 'val', 'test']
TMP_IMGS = BASE / '_all_images'
TMP_LBLS = BASE / '_all_labels'
TMP_IMGS.mkdir(exist_ok=True)
TMP_LBLS.mkdir(exist_ok=True)

# Collect every image+label pair across all existing splits
pairs = []
for split in SPLITS:
    img_dir = BASE / split / 'images'
    lbl_dir = BASE / split / 'labels'
    if not img_dir.exists():
        continue
    for img in img_dir.glob('*.*'):
        lbl = lbl_dir / (img.stem + '.txt')
        if lbl.exists():
            pairs.append((img, lbl))

print(f'Total image-label pairs found: {len(pairs)}')

# Move everything to tmp
for img, lbl in pairs:
    shutil.move(str(img), TMP_IMGS / img.name)
    shutil.move(str(lbl), TMP_LBLS / lbl.name)

# Shuffle and split 70/15/15
random.seed(42)
random.shuffle(pairs)
n = len(pairs)
n_train = int(n * 0.70)
n_val   = int(n * 0.15)

split_map = {
    'train': pairs[:n_train],
    'val':   pairs[n_train:n_train + n_val],
    'test':  pairs[n_train + n_val:],
}

# Recreate split folders and move files
for split, split_pairs in split_map.items():
    img_dir = BASE / split / 'images'
    lbl_dir = BASE / split / 'labels'
    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)
    for img, lbl in split_pairs:
        shutil.move(str(TMP_IMGS / img.name), img_dir / img.name)
        shutil.move(str(TMP_LBLS / lbl.name), lbl_dir / lbl.name)

# Cleanup tmp dirs
TMP_IMGS.rmdir()
TMP_LBLS.rmdir()

for split in SPLITS:
    count = len(list((BASE / split / 'images').glob('*.*')))
    print(f'{split:5}: {count} images')

In [ ]:
# 6. Write data.yaml
import yaml

data_yaml = {
    'path': str(BASE),
    'train': 'train/images',
    'val':   'val/images',
    'test':  'test/images',
    'nc': 1,
    'names': {0: 'toddler'}
}

yaml_path = BASE / 'data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

print('data.yaml written to', yaml_path)

In [ ]:
# 6b. Download adult-only hard negatives from Roboflow (Children vs Adults dataset)
!pip install roboflow -q

from roboflow import Roboflow
import shutil, yaml
from pathlib import Path

RF_API_KEY   = "nJdoWfJScD3E65immFfC"
HARD_NEG_DIR = Path('/content/hard_negatives')
BASE         = Path('/content/dataset/detect_child')
DEST_IMGS    = BASE / 'train' / 'images'
DEST_LBLS    = BASE / 'train' / 'labels'

# Download Children vs Adults dataset
rf      = Roboflow(api_key=RF_API_KEY)
project = rf.workspace("a-4euhx").project("children-vs-adults-yolo-my3ct")
dataset = project.version(4).download("yolov8", location=str(HARD_NEG_DIR))

# Read class names from downloaded dataset yaml
with open(HARD_NEG_DIR / 'data.yaml') as f:
    meta = yaml.safe_load(f)

names = meta['names']
print("Classes in downloaded dataset:", names)

# Find child class index(es)
if isinstance(names, dict):
    child_ids = {k for k, v in names.items() if 'child' in str(v).lower()}
else:
    child_ids = {i for i, v in enumerate(names) if 'child' in str(v).lower()}
print(f"Child class IDs to exclude: {child_ids}")

# Add adult-only images as hard negatives (empty label files)
added = 0
for split in ['train', 'valid', 'test']:
    img_dir = HARD_NEG_DIR / split / 'images'
    lbl_dir = HARD_NEG_DIR / split / 'labels'
    if not img_dir.exists():
        continue
    for img_path in img_dir.glob('*.*'):
        lbl_path = lbl_dir / (img_path.stem + '.txt')
        has_child = False
        if lbl_path.exists():
            for line in lbl_path.read_text().strip().splitlines():
                if line.strip() and int(line.split()[0]) in child_ids:
                    has_child = True
                    break
        if not has_child:
            dest_img = DEST_IMGS / img_path.name
            if not dest_img.exists():
                shutil.copy(str(img_path), dest_img)
                (DEST_LBLS / (img_path.stem + '.txt')).write_text('')
                added += 1

print(f"\nAdded {added} adult-only hard negative images to train set")
print(f"New train total: {len(list(DEST_IMGS.glob('*.*')))} images")

In [ ]:
# 6c. Add your own adult hard negatives from Google Drive
import zipfile, shutil
from pathlib import Path

ADULT_NEG_ZIP = '/content/drive/MyDrive/adult_negatives.zip'
ADULT_NEG_DIR = Path('/content/adult_negatives')
BASE          = Path('/content/dataset/detect_child')
DEST_IMGS     = BASE / 'train' / 'images'
DEST_LBLS     = BASE / 'train' / 'labels'

with zipfile.ZipFile(ADULT_NEG_ZIP, 'r') as z:
    z.extractall(str(ADULT_NEG_DIR))

added = 0
for img_path in ADULT_NEG_DIR.rglob('*.jpg'):
    dest_img = DEST_IMGS / img_path.name
    if not dest_img.exists():
        shutil.copy(str(img_path), dest_img)
        (DEST_LBLS / (img_path.stem + '.txt')).write_text('')
        added += 1

print(f"Added {added} custom adult hard negatives to train set")
print(f"New train total: {len(list(DEST_IMGS.glob('*.*')))} images")

In [ ]:
# 7. Train
from ultralytics import YOLO
from pathlib import Path

yaml_path = Path('/content/dataset/detect_child/data.yaml')

model = YOLO('yolov8s.pt')   # small model — more accurate than nano

results = model.train(
    data=str(yaml_path),
    epochs=100,
    imgsz=640,
    batch=64,          # H100 / A100 can handle 64
    patience=15,
    save=True,
    plots=True,
    project='/content/runs',
    name='toddler_detect_s100',
    exist_ok=True,
)

In [ ]:
# 8. Evaluate on test set
metrics = model.val(
    data=str(yaml_path),
    split='test',
    project='/content/runs',
    name='toddler_detect_s100_test',
)
print(f'mAP50:    {metrics.box.map50:.4f}')
print(f'mAP50-95: {metrics.box.map:.4f}')

In [ ]:
# 9. Save best weights back to Google Drive
import shutil

SAVE_TO = '/content/drive/MyDrive/toddler_detect_s100_best.pt'
shutil.copy('/content/runs/toddler_detect_s100/weights/best.pt', SAVE_TO)
print('Saved to Google Drive:', SAVE_TO)